# Week 10 - Day 3 Lab
# AI Agent Frameworks with LangChain & LangGraph

---

## Learning Objectives

By the end of this lab, you will be able to:

- Build an AI Agent using LangChain.
- Register and use custom tools.
- Understand how LangChain simplifies the manual agent loop.
- Build a simple workflow using LangGraph.
- Add memory to an AI Agent.
- Extend the agent with your own custom tool.

Estimated Time: 2.5 Hours

# Step 1: Install Required Libraries

Run the following cell to install all required libraries.

In [1]:
!pip -q install -U langchain
!pip -q install -U langgraph
!pip -q install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 26.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


# Step 2: Configure Gemini

We will use Google's Gemini model throughout this lab.

In [7]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_Key")
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

Key loaded: True


## Create the Gemini LLM

Import the required class and initialize the model.

💡 Hint

The class name starts with

ChatGoogle...

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)


### Test Your Model

If everything is configured correctly, the model should respond.

Expected Output

Hello! How can I help you today?

In [10]:
response = llm.invoke("Say Hello!")

print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPf6070fkMOXbt5ekH+HwY0DQpgLHeq9SyFbEOaYWDZ/6fEe5OimW/j4gtS+Xzf/kZtHy4kk2UfAYJ1XweVbdveWG6lKmau11HDzrp+JiPYlqTkWRhn0LX'}}]


# Creating Your First Tool

A tool is simply a Python function decorated with `@tool`.

Today we'll build a Weather Tool.

## Task

Complete the dictionary below.

💡 Hint

The dictionary key should be the city name.

The value should be the weather.

In [12]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Returns the weather of a city.
    """

    weather = {
        "Lahore": "Sunny, 34°C",
        "Karachi": "Humid, 31°C",
        "Islamabad": "Cloudy, 27°C"
    }

    return weather.get(city, "Weather unavailable")

## Test Your Tool

In [13]:
print(get_weather.invoke("Lahore"))

Sunny, 34°C


## Challenge

Add TWO more cities.

Suggested cities:

- Peshawar
- Quetta

In [14]:
@tool
def get_weather(city: str) -> str:
    """
    Returns the weather of a city.
    """

    weather = {
        "Lahore": "Sunny, 34°C",
        "Karachi": "Humid, 31°C",
        "Islamabad": "Cloudy, 27°C",
        "Peshawar": "Hot, 36°C",
        "Quetta": "Cool, 22°C"
    }

    return weather.get(city, "Weather unavailable")

In [15]:
print(get_weather.invoke("Peshawar"))
print(get_weather.invoke("Quetta"))

Hot, 36°C
Cool, 22°C


# Calculator Tool

Let's create another tool.

## Task

Complete the function.

💡 Hint

Python has a function that evaluates mathematical expressions.

In [16]:
@tool
def calculator(expression: str):
    """
    Evaluate a mathematical expression.
    """

    return eval(expression)

## Test

Try

25*18

Expected Output

450

In [17]:
calculator.invoke("25*18")

450

# Registering Tools

The AI Agent needs to know which tools it can use.

## Task

Complete the list below.

💡 Hint

Use the function names only.

In [18]:
tools = [
    get_weather,
    calculator
]

# Binding Tools to Gemini

Gemini must know which tools are available.

## Task

Complete the following line.

💡 Hint

The method starts with

bind...

In [19]:
llm_with_tools = llm.bind_tools(tools)

## Test Tool Calling

Ask Gemini about the weather.

Observe the response carefully.

Does Gemini answer directly?

Or

Does it request a tool?

In [20]:
response = llm_with_tools.invoke(

    "What's the weather in Lahore?"

)

response

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Lahore"}'}, '__gemini_function_call_thought_signatures__': {'call_427853': 'El4KXAERTTIP4S4sHUhmXNx5KZ7um883y+X9Qz+ooTl6JBWY2H18na2vgHu7oPSagS69HCSynr402J6WcKpnBlBIG5bxf0CIRfxgIG13c0JcuYFw6lI67brsIjzSCful'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06789-6b1f-71e3-ac1b-36944a477bb2-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': 'call_427853', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 17, 'total_tokens': 110, 'input_token_details': {'cache_read': 0}})

# Tool Registry

When Gemini requests a tool, Python needs to know which function to execute.

## Task

Complete the registry.

💡 Hint

The key is a string.

The value is the function.

In [21]:
tool_registry = {
    "get_weather": get_weather,
    "calculator": calculator
}

# Executing Tool Calls

Complete the missing lines.

💡 Hint

1. Find the tool.

2. Execute it.

3. Print the result.

In [22]:
for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool = tool_registry[tool_name]
    result = tool.invoke(args)

    print(result)

Sunny, 34°C


# Creating a LangChain Agent

Earlier we manually:

- Parsed tool calls
- Executed tools
- Returned observations

LangChain automates all of that.

Complete the missing imports.

💡 Hint

The required classes are:

- AgentExecutor
- create_tool_calling_agent

In [24]:
!pip -q install -U langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.2 MB/s eta 0:00:00


In [25]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

## Build the Agent

Fill in the blanks.

Don't worry if you don't remember every function—we discussed this in today's lecture.

💡 Hint

Use:

- llm
- tools
- prompt

In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [27]:
agent = create_tool_calling_agent(llm, tools, prompt)

In [28]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## Test the Agent

In [29]:
result = agent_executor.invoke({

    "input":"Should I carry an umbrella in Islamabad?"

})

print(result["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Islamabad'}`


Cloudy, 27°C

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "It is currently cloudy and 27°C in Islamabad. While it's not actively raining, the cloudy conditions mean there's a chance of precipitation, so carrying an umbrella might be a good idea just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIP2dlaCUHzU2Kr2sJKoe7NrKHjXO5z4xq7NGLCESjq0bEHsN2c+W7nsfTgtd+Ebti5B8YF+dmb78N4em2poXniSPm6ONMKqDtFrFHDzQez6i7e6xjzYxD1'}}]

> Finished chain.
[{'type': 'text', 'text': "It is currently cloudy and 27°C in Islamabad. While it's not actively raining, the cloudy conditions mean there's a chance of precipitation, so carrying an umbrella might be a good idea just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIP2dlaCUHzU2Kr2sJKoe7NrKHjXO5z4xq7NGLCESjq0bEHsN2c+W7nsfTgtd+Ebti5B8YF+dmb78N4em2poXniSPm6ONMKqDtFrFHDzQez6i7e6xjzYxD1'}}]


# LangGraph State

LangGraph stores information inside a shared State object.

## Task

Complete the state.

💡 Hint

Today's slides showed three important fields.

One of them is

messages

In [30]:
from typing import TypedDict

class AgentState(TypedDict):
    messages: list
    city: str
    output: str

# Creating Nodes

Every node is simply a Python function.

Complete the chatbot node.

💡 Hint

The node should

1. Read the state

2. Call the LLM

3. Return the updated state

In [31]:
def chatbot_node(state):
    response = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

# Memory

Let's build a very small conversation memory.

Complete the class.

💡 Hint

Remember:

append()

returns nothing.

In [32]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)

    def get_context(self):
        return self.messages

## Challenge

Modify your memory so it only stores the latest **5** messages.

Hint:

Python list slicing may be useful.

In [33]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)
        self.messages = self.messages[-5:]

    def get_context(self):
        return self.messages

# Smart Travel Assistant

Congratulations!

You now know how to build AI Agents using LangChain.

## Your Task

Build a Travel Assistant.

Requirements

✅ Weather Tool

✅ Calculator Tool

Create ONE new tool.

Choose ONE:

- Restaurant Tool
- Movie Tool
- Hotel Tool
- Currency Converter

The assistant should answer questions like:

"I'm visiting Lahore tomorrow.

What's the weather?

Recommend a restaurant.

How much will dinner cost for 4 people if each meal costs $25?"

---

## Bonus Challenge

Can your assistant remember the user's city without asking again?

Example

User:

"I'm visiting Lahore."

Later...

"What's the weather tomorrow?"

The assistant should understand that the city is still Lahore.

In [35]:
# write code here
from langchain.tools import tool

@tool
def get_restaurant(city: str) -> str:
    """
    Recommends a restaurant in a given city.
    """

    restaurants = {
        "Lahore": "Cafe Aylanto",
        "Karachi": "Kolachi",
        "Islamabad": "Monal",
        "Peshawar": "Shiraz Chinese Restaurant",
        "Quetta": "Lakpal Hotel"
    }

    return restaurants.get(city, "No recommendation available")

In [36]:
travel_tools = [get_weather, calculator, get_restaurant]

travel_tool_registry = {
    "get_weather": get_weather,
    "calculator": calculator,
    "get_restaurant": get_restaurant
}

In [37]:
travel_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Use tools when needed to answer questions about weather, restaurants, or costs."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

travel_agent = create_tool_calling_agent(llm, travel_tools, travel_prompt)
travel_agent_executor = AgentExecutor(agent=travel_agent, tools=travel_tools, verbose=True)

In [38]:
result = travel_agent_executor.invoke({
    "input": "I'm visiting Lahore tomorrow. What's the weather? Recommend a restaurant. How much will dinner cost for 4 people if each meal costs $25?"
})
print(result["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Lahore'}`


Sunny, 34°C
Invoking: `get_restaurant` with `{'city': 'Lahore'}`


Cafe Aylanto
Invoking: `calculator` with `{'expression': '4 * 25'}`


100

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here is the information for your trip to Lahore:\n\n* **Weather:** Sunny and 34°C. \n* **Restaurant Recommendation:** Cafe Aylanto\n* **Dinner Cost:** For 4 people at $25 per meal, the total cost will be $100.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPXftelh7jhRvITpd/h90WfQXPAKhGOjZbH0JM8FjywnsuXvQKAXOtwcCol8OJn+3BSbfseVVcByFhtVJ+jL6vUd5eGV/nC5R54u6c/HObM0k+POZ4mymh'}}]

> Finished chain.
[{'type': 'text', 'text': 'Here is the information for your trip to Lahore:\n\n* **Weather:** Sunny and 34°C. \n* **Restaurant Recommendation:** Cafe Aylanto\n* **Dinner Cost:** For 4 people at $25 per meal, the total cost will be $100.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPXftelh7jhRvITpd/h90WfQXPAKhGOjZbH0JM8FjywnsuXvQKAXOtwcCol8OJn+3BSbfseVVcByFhtVJ+jL6vUd5eGV/nC5R54u6c/HObM0k+POZ4mymh'}}]


# Bonus Challenge

In [39]:
chat_history = []

def ask_travel_assistant(user_input):
    result = travel_agent_executor.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    chat_history.append(("human", user_input))
    chat_history.append(("ai", result["output"]))
    return result["output"]

In [40]:
travel_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Use tools when needed to answer questions about weather, restaurants, or costs. Remember details the user has already told you, like their city."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

travel_agent = create_tool_calling_agent(llm, travel_tools, travel_prompt)
travel_agent_executor = AgentExecutor(agent=travel_agent, tools=travel_tools, verbose=True)

In [41]:
print(ask_travel_assistant("I'm visiting Lahore."))
print(ask_travel_assistant("What's the weather tomorrow?"))



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'That sounds wonderful! Lahore is a city with an incredible history, amazing architecture, and world-famous food. \n\nAre you looking for recommendations on where to eat, what the weather is going to be like, or anything else to help plan your trip?', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPiY+N0nSJeSDF6/37c7bQosP5WCD2A+rZgrFn5H/t7xfB0hf4o7KGOeMn6A9zbwr/6+NhNWO8dDevklW44QO0mniCdYiE3UhupH4Tr/jsa2P2ok07tt1f'}}]

> Finished chain.
[{'type': 'text', 'text': 'That sounds wonderful! Lahore is a city with an incredible history, amazing architecture, and world-famous food. \n\nAre you looking for recommendations on where to eat, what the weather is going to be like, or anything else to help plan your trip?', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPiY+N0nSJeSDF6/37c7bQosP5WCD2A+rZgrFn5H/t7xfB0hf4o7KGOeMn6A9zbwr/6+NhNWO8dDevklW44QO0mniCdYiE3UhupH4Tr/jsa2P2ok07tt1f'}}]


> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Lahore'}`


Sunny, 34°C

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "Tomorrow in Lahore, it's going to be sunny with a high of 34°C. Don't forget your sunglasses and some sunscreen if you plan on heading out!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIP+UsF3DiFE++iVc+tlVlVajxXcu8LFf7VjMi8Mz+i8iQrM6A/gTR7Uh7L97i0/zkJkaDsCMvn4q45VY4+6oFrzzZ0CvzcrJsUnXG6535C7525N+4kcNko'}}]

> Finished chain.
[{'type': 'text', 'text': "Tomorrow in Lahore, it's going to be sunny with a high of 34°C. Don't forget your sunglasses and some sunscreen if you plan on heading out!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIP+UsF3DiFE++iVc+tlVlVajxXcu8LFf7VjMi8Mz+i8iQrM6A/gTR7Uh7L97i0/zkJkaDsCMvn4q45VY4+6oFrzzZ0CvzcrJsUnXG6535C7525N+4kcNko'}}]
